# 8주차 · 비행안전 · IIP · FTS · 임무보증
### 우주발사체는 왜 '자폭 장치'가 필요한가 — 순시낙하점(IIP)이 파괴선을 넘는 순간부터 카운트다운이 시작된다

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabraxas/LVs-and-Policy/blob/main/lecture/week08/week08.ipynb)


In [ ]:
# ▶ 실행 전 준비 — 이 셀을 먼저 실행하세요 (약 10~20초 소요)
!pip install -q ipywidgets
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/_shared/course_interactive.py",
    "course_interactive.py")

from course_interactive import *
setup_korean_font()
print("준비 완료 — 아래 셀들을 순서대로 실행하며 강의를 진행하세요.")


## 학습 목표

- [ ] **FTS의 3대 구성요소**를 설명하고, 지상 CTS → RF 업링크 → 탑재 FTR → SAD → 파괴장치 흐름을 그릴 수 있다
- [ ] **IRIG 톤 명령 방식**을 이해하고, 420~450 MHz 대역과 3~4톤 조합이 왜 사용되는지 설명한다
- [ ] **IIP(순시낙하점)**를 정의하고, 비반복형 케플러 알고리즘이 어떻게 실시간 연산을 가능하게 하는지 설명한다
- [ ] **도그레그 기동**의 원리와 페이로드 페널티를 설명하고, 발사장 위치·궤도 선택과의 연관성을 논한다
- [ ] **Ec(예상인명피해)** 수식을 이용해 발사 허가 기준 준수 여부를 판단할 수 있다
- [ ] **나로호→누리호 FTS 진화**를 통해 안전 설계와 정책·국제 환경이 교차하는 지점을 설명한다


## 오늘의 4부 구성

| 부 | 주제 | 핵심 질문 |
|---|---|---|
| **1부** | FTS 구조·통신 방식 | 로켓엔 왜 자폭 장치가 있고, 어떻게 신호를 보내는가? |
| **2부** | IIP — 순시낙하점의 과학 | 지금 엔진이 꺼지면 발사체는 어디 떨어지나? |
| **3부** | 낙하점 회피 — 도그레그·VIP | 안전한 궤적을 능동적으로 설계하는 방법 |
| **4부** | Ec·임무보증·한국 사례 | 얼마나 안전해야 발사 허가를 받는가? |


---
## PART 1 · 비행종단시스템(FTS) — 구조와 통신 방식


### FTS란 무엇인가

> **핵심 정의**: 발사체가 정상 궤적을 이탈해 공공 안전을 위협할 때, 지상 또는 자율 판단으로 비행을 강제 종료시키는 탑재 서브시스템

**직관적 비유**: 도심을 달리는 버스에 비상 브레이크가 있듯, 발사체에는 "비상 정지" 장치가 있어야 한다.  
단, 이 브레이크는 **정해진 암호(IRIG 톤 조합)** 를 받아야만 작동하고, 신호가 끊기면 **작동하지 않는 것이 안전** (Fail-Safe).

**미국 법적 근거**: Range Commanders Council **RCC 319** — 미국 비행시험장을 비행하는 모든 발사체에 FTS 의무화


#### 🔵 아래 셀을 실행하면 FTS 전체 구조가 그려집니다


In [ ]:
fts_system_diagram()

### 통신 방식 심화 — IRIG 톤 명령

FTS는 단순히 "신호 있음/없음"이 아닌, **지정된 오디오 톤(가청 주파수) 여러 개를 FM 반송파에 실어** 보내는 방식을 사용한다.

| 구분 | 내용 |
|---|---|
| **반송파 주파수** | **420 ~ 450 MHz (UHF)** — 회절·항공주파수 이격·도플러 최소화의 균형 |
| **변조 방식** | **FM (주파수 변조)** — 반송파 위에 오디오 톤 신호 탑재 |
| **명령 인코딩** | **IRIG 313 표준**: 20개 기준 톤 주파수 중 3~4개 조합으로 명령 구분 |
| **명령 시퀀스** | **SAFE → ARM → TERMINATE** (2단계 확인 — 오발 확률 극소화) |
| **Fail-Safe 원칙** | 신호 두절 시 → **기폭 안 됨** (일반 잠금장치의 반대 개념) |

**왜 톤 조합인가?** 단일 신호로는 전파 잡음·우주방사선 비트 플립 등으로 오작동 위험.  
정해진 3~4개 톤이 동시에 정확한 순서로 들어와야 명령 실행 → 오발 확률 극적으로 감소.


#### 📡 아래 셀을 실행하면 IRIG 톤 통신 방식을 인터랙티브하게 탐색할 수 있습니다


In [ ]:
irig_tone_explorer()

### 핵심 제품 — Curtiss-Wright FTR-200

| 사양 | 값 |
|---|---|
| 수신 주파수 | 420 ~ 450 MHz (프로그래머블) |
| 수신 감도 | -107 dBm ~ +13 dBm 동적 범위 |
| 톤 디코딩 | 3 또는 4톤 (20개 IRIG 톤 중 선택) |
| 크기 | **3.7 in³** (초소형 탑재) |
| 전원 | 22 ~ 36 VDC |
| 준거 표준 | **RCC-319-10**, MIL-STD-810G |

**SSTO(신호강도 텔레메트리 출력)**: 수신기가 실시간으로 신호 강도를 지상에 알려줌 → 지상에서 FTR 작동 상태 확인 가능

**안전·기폭장치(ESAD)** — PacSci EMC 대표 제품:
- 6가지 RCC 319 인증 독립 설계 라인업
- FPGA(Actel Anti-Fuse) — 방사선에 의한 비트 플립 원천 차단
- 허메틱(Hermetic) 밀봉 — 고고도·우주 환경 대응
- Atlas, Delta, Antares 등 주요 미국 발사체 납품 실적


---
## PART 2 · 순시낙하점(IIP) — 비행안전 판단의 핵심 지표


### IIP란 무엇인가

> **IIP(Instantaneous Impact Point, 순시낙하점)**: 발사체의 추진 기관이 즉각 작동을 멈춘다고 가정할 때, **현재 위치·속도 벡터만으로** 산출되는 지표면 상의 예상 낙하 위치

```
발사체 정상 비행 중 →  IIP가 지표면을 따라 목표 궤도 방향으로 전진
                        ↓
IIP가 사전 설정 파괴선(Destruct Line) 침범
                        ↓
비행종단 결정 → FTS 작동
```

**비유**: IIP = "지금 핸들을 놓으면 차가 어디로 가는가"  
RSO(Range Safety Officer)는 그 지점이 인구 밀집 지역 방향이면 즉시 FTS를 작동시킨다.

**핵심 도전**: 실시간 비행 통제 소프트웨어는 **수십 ms 이내**에 IIP를 계산해야 한다.  
6-DOF 수치 적분은 정확하지만 너무 느림 → **해석적(Analytic) 알고리즘** 필수


### IIP 4세대 알고리즘 비교

| 세대 | 모델 | 물리 수준 | 오차 (장거리) | 실시간 부하 |
|---|---|---|---|---|
| **1세대** | 평면지구 포물선 | 상수 중력 + 1차 항력 | ~45 km | 극히 낮음 |
| **2세대** | 반복형 케플러 | 역제곱 중력 + 타원체 | ~8 km | 중간 (특이점 발산 위험) |
| **3세대** | **비반복형 케플러 (Ahn & Roh, 2012)** | 역제곱 + 타원체 보정 | **~2.5 km** | **극히 낮음** |
| **4세대** | **RSM 하이브리드** | 케플러 + 다항식 항력 보정 | **~0.4 km** | **극히 낮음** |

> 3세대 비반복형 케플러: 반복 연산 없이 **2.5배** 이상 속도 향상, 특이점 발산 위험 제거  
> 4세대 RSM: 비행 전 몬테카를로 시뮬레이션으로 항력 오차를 다항식으로 학습 → 비행 중 대입만


#### 📊 아래 셀을 실행하면 4세대 알고리즘을 오차·연산 부하로 비교합니다


In [ ]:
iip_algorithm_comparison()

### IIP 계산 핵심 수식

**비행각(φ) 계산식** — Ahn & Roh (2012) 비반복형:

$$\cos(\phi) = \frac{1 - \lambda\cos^2\gamma_0 \pm \sqrt{(1-\lambda\cos^2\gamma_0)^2 - \left(1-\frac{r_0}{r_p}\lambda\cos^2\gamma_0\right)\cos^2\gamma_0}}{1 - \frac{r_0}{r_p}\cos^2\gamma_0}$$

- $\gamma_0$: 현재 비행 경로각 | $\lambda = v_0^2/(\mu/r_0)$: 속도 에너지 비율 | $r_p$: 지표 반경

**IIP 위치 벡터**:
$$\mathbf{p}_I = \frac{\cos(\phi - \gamma_0)}{\cos\gamma_0}\mathbf{i}_{r0} + \frac{\sin\phi}{\cos\gamma_0}\mathbf{i}_{v0}$$

**지구 자전 보정** (체공시간 동안 지구가 돌아감):
$$Lon^E_p = Lon^I_p - \omega_e \cdot t_F \quad (\omega_e = 7.29 \times 10^{-5} \text{ rad/s})$$


#### 🗺️ 아래 셀을 실행하면 속도·경로각·고도를 바꾸며 IIP 궤적을 실시간으로 볼 수 있습니다


In [ ]:
iip_trajectory_explorer()

---
## PART 3 · 낙하점 회피 전략 — 도그레그 기동과 VIP


### 도그레그(Dogleg) 기동 — 능동적 IIP 회피

**왜 필요한가?**  
연료 최적 경로(일직선)가 인구 밀집지·타국 영토를 지나는 경우,  
발사 실패 시 IIP가 해당 지역에 떨어지는 문제가 생긴다.

**원리**:
- 상승 초기 **방위각(Azimuth)을 급격히 변경** — 초당 약 10°의 요 스티어링(Yaw Steering)
- 궤적이 "개 뒷다리" 모양으로 꺾임 → 무인 해상 구역으로 IIP 유도

**케이프 커내버럴(위도 28.5°N) 극궤도 발사 예시**:
```
수직 상승 → 방위각 170° (남쪽) → 횡방향 10~90° 꺾기
                    ↓
미국 본토·남미 인구밀집 지역 우회
                    ↓
실패 시에도 IIP가 무인 해상 구역에 머뭄
→ Ec ≤ 1×10⁻⁴ 기준 준수
```

**⚠️ 페널티**: 비수직 상승 + 궤도면 변경 → 추가 ΔV 소모  
소형 발사체는 페이로드 용량에 치명적 → **도그레그 각도가 곧 비즈니스 문제**


#### ✈️ 아래 셀을 실행하면 도그레그 각도·시작 시간을 조절해 IIP 궤적과 페널티를 비교할 수 있습니다


In [ ]:
dogleg_maneuver_explorer()

### IIP 유지 유도법칙 — 재사용 발사체(RLV)에의 적용

**Falcon 9 부스트백(Boost-back Burn)**: 1단이 착륙장으로 귀환하는 궤적 설계는 사실상 **IIP 제어 문제**

**IIP Hold Guidance**:
- 엔진 컷오프 실패·추력 저하 등 악조건에서도 IIP가 **착륙장 주변을 유지**하도록 피드백 제어
- 비행 경로각 변화율 제어 + 최소 임펄스 유도(ΔV 억제) 서브루틴 결합
- IIP 미분식($d\mathbf{i}_p/dt$)을 피드백 루프에 직접 투입 → 수치 적분 없이 즉각 명령 생성

### VIP(Virtual Impact Point, 가상낙하점) — IIP의 방어 적용

> **VIP**: 탄도 표적이 낙하할 **예상 지점** — 요격 미사일의 중간 유도 기준점

| 개념 | 적용 분야 | 역할 |
|---|---|---|
| **IIP** | 발사체 비행안전 | "내가 어디 떨어지나" → 파괴선 초과 시 FTS 작동 |
| **VIP** | 미사일 방어(BMD) | "표적이 어디 떨어지나" → 요격 미사일이 그 지점으로 비행 |

**동일한 수학**이 비행안전(낙하점 **회피**)과 미사일 방어(낙하점 **추적**)에 공통 적용됨


---
## PART 4 · 예상인명피해(Ec) · 임무보증 · 한국 사례


### Ec(예상인명피해) 확률 모델

발사 허가의 핵심 수치적 기준:

$$E_c = \sum_{j} P_{f,j} \sum_{i} \left( P_{impact,i} \times A_{c,i} \times D_{p,i} \right)$$

| 기호 | 의미 |
|---|---|
| $P_{f,j}$ | 발사체 고도별 오작동·고장 발생 확률 |
| $P_{impact,i}$ | 파편이 지상 타겟 $i$에 낙하할 확률 밀도 |
| $A_{c,i}$ | 사상 면적 (파편 관통력·폭발파·인체 취약성 모두 포함) |
| $D_{p,i}$ | 인구 밀도 (시간대·실내외 비율까지 반영) |

**미국 FAA 14 CFR §450 현행 기준**:

| 지표 | 대상 | 기준 |
|---|---|---|
| 집단 Ec | 일반 대중 | ≤ **1×10⁻⁴** (만 명 당 1명) |
| 개인 Pc | 일반인 | ≤ **1×10⁻⁶** (백만 명 당 1명) |
| 항공기 충돌 | 공역 항공기 | < **1×10⁻⁶** |

> 과거: 폭발·독성·과압 각각 $30\times10^{-6}$ 개별 적용 → 2010년 이후 통합 한계 $1\times10^{-4}$로 일원화


#### 📊 아래 셀을 실행하면 각 인자를 조절해 Ec를 계산하고 발사 허가 기준을 확인할 수 있습니다


In [ ]:
ec_risk_calculator_w8()

### 나로호→누리호 FTS 진화 — 국내 사례

**나로호 2차 발사(2010) 폭발 — FTS 오작동 가설**:
- 이륙 후 137초경 고공 폭발
- 유력 원인: 2단 FTS의 **예기치 않은 오작동** → 킥모터 고체 추진제 연소·폭발
- **전통적 FTS의 역설**: 안전장치(뇌관+폭약)가 오히려 **실패 트리거**가 될 수 있음

**누리호(KSLV-II)의 해법**:
- 화약 폭발형 FTS 최소화 → **산화제·연료 밸브 차단** (엔진 정지) 방식 채택
- 오작동 트리거 위험 제거 + ICBM 기술 전용 의혹 탈피 + 100% 국산화


#### 🇰🇷 아래 셀을 실행하면 나로호→누리호 FTS 진화 타임라인과 비교표를 볼 수 있습니다


In [ ]:
kslv_fts_evolution()

### 임무보증(Mission Assurance) — 실패를 막는 프로세스

**Aerospace Corporation 6대 핵심 프로세스**:
1. **요구사항 분석·검증** — 고객 요구와 설계 사양의 일치 확인
2. **설계 검토** — PDR(예비), CDR(상세) 단계별 독립 검토
3. **제작·시험 검증** — 부품 수준부터 시스템 수준까지
4. **발사 준비태세 검토(RDR)** — 발사 직전 최종 확인
5. **이상 대응** — 문제 발생 시 원인 분석·시정 조치
6. **교훈 반영** — 실패를 다음 임무에 반영

**임무보증 실패 사례 — Mars Climate Orbiter (1999)**:
- Lockheed Martin: 추력 데이터를 **lbf·s** (야드파운드법)으로 산출
- NASA JPL: **N·s** (SI)로 오인 → **4.45배** 오차 누적
- 화성 대기권 진입 고도 오류 → $1.25억 달러 우주선 소실

> **TRANSCOST와 같은 유형**: 톤(t) vs kg 단위 버그 → 우주공학에서 **단위 불일치는 가장 흔한 사고 원인**


#### 🔄 아래 셀을 실행하면 임무보증 프레임워크와 MCO 실패 사례를 함께 볼 수 있습니다


In [ ]:
mission_assurance_timeline()

### 한국 나로우주센터 민간 발사 안전 체계 (2026~)

| 조치 | 내용 |
|---|---|
| **Ec 공식 채택** | 발사 허가 핵심 기준으로 Ec 명시 |
| **임시비행금지구역** | 발사 1일 전부터 반경 3km, 드론·촬영 원천 통제 |
| **해상통제구역** | IIP 예상 해상 구역 선박 접근 차단 |
| **발사안전통제협의회** | 고압가스·화학위험물 관리 체계 감시 |
| **책임보험 의무화** | 민간기업 발사 실패 시 배상 담보 |

> 7주차에서 다룬 **우주손해배상법** · **발사허가 체계** + 이번 주 Ec 기준이 결합하는 지점  
> → 발사 안전 규제는 **공학(IIP·Ec) + 법률(허가·보험) + 정치(ICBM 의혹)**가 교차하는 복합 체계


---
## 핵심 개념 대조표 — 시험·캡스톤 작성용

| 개념 | 정의 | 비행안전에서의 역할 |
|---|---|---|
| **IIP** (순시낙하점) | 엔진 즉시 정지 시 예상 낙하 지점 | FTS 판단 기준점. 파괴선 침범 시 종단 |
| **VIP** (가상낙하점) | 탄도 표적의 예상 낙하 지점 | 요격 미사일 중간 유도. IIP의 방어 버전 |
| **Destruct Line** (파괴선) | IIP가 넘으면 FTS 작동하는 경계 | 발사 전 Ec 기준으로 결정 |
| **도그레그** | 방위각 급변으로 궤적을 꺾는 기동 | IIP를 무인 구역으로 능동 유도 |
| **IRIG 톤** | 20개 기준 오디오 주파수 중 3~4개 조합 | FTS 명령 인코딩. 오발 방지 |
| **Fail-Safe** | 신호 두절 시 파괴 안 됨 | FTS 핵심 설계 원칙 |
| **Ec** | 발사 당 예상 사상자 수 기댓값 | 발사 허가 기준. FAA ≤ 1×10⁻⁴ |
| **RSM** (응답면 기법) | 항력 오차를 다항식으로 사전 학습 | 실시간 IIP 정밀도 향상 |
| **ESAD** | 전자식 안전·기폭장치 | Safe→Arm→Fire 시퀀스 실행 |
| **LSC** | 선형정형작약 | 발사체 동체 절단으로 추력 제거 |


---
## 실습 워크숍 · 우리 팀 발사체의 비행안전 체크리스트

**팀별 30분** — 캡스톤 보고서 '리스크 분석' 섹션 초안 작성

1. **IIP 회랑 설계**: 나로우주센터에서 발사할 경우 IIP가 일본·중국 영토를 피하려면 어떤 방위각 제약이 생기는가?
2. **도그레그 필요성 검토**: 팀 발사체의 목표 궤도가 SSO(태양동기궤도)라면 도그레그가 필요한가? ΔV 페널티를 개략 추산하라.
3. **Ec 예비 계산**: 위의 Ec 계산기를 이용해 팀 발사체의 가상 시나리오(발사 실패 확률 = 3%)에서 Ec를 계산하고, FAA 기준과 비교하라.
4. **FTS 방식 선택**: 팀 발사체에 폭발형 FTS를 사용할 것인가, 엔진 차단 방식을 사용할 것인가? 각각의 장단점을 정리하라.


---
## 참고문헌 및 학습자료

**IIP 이론**
- Ahn, J. & Roh, W. (2012). *Noniterative Instantaneous Impact Point Prediction Algorithm*
- Jo, S. & Ahn, J. (2017). *Geometric derivation of IIP time derivatives*
- Ahn, J. & Seo, S. (2013). *Response-Surface Method for IIP correction*

**FTS 표준 및 제품**
- RCC 319-25 (FTS 공통화 표준, 2025.6): https://www.trmc.osd.mil/wiki/download/attachments/113019893/319-25_FTS_Commonality.pdf
- IRIG 313-24 (FTR 시험 표준): https://www.trmc.osd.mil/wiki/download/attachments/113019889/313-24.pdf
- PacSci EMC FTS 제품: https://psemc.com/products/flight-termination-system/
- Curtiss-Wright FTR-200: https://www.curtisswrightds.com/products/flight-test/radio-frequency/ftr200
- SIL VBITS AFTS: https://www.spaceinformationlabs.com/products/gps-tracking-and-afts/

**규제 기준**
- FAA 14 CFR §450.101: https://www.ecfr.gov/current/title-14/chapter-III/subchapter-C/part-450
- National Academies — Streamlining Space Launch Range Safety: https://www.nationalacademies.org/read/9790
- FAA — Starship Mishap Investigation: https://www.faa.gov/newsroom/faa-closes-spacex-starship-mishap-investigation

**국내 사례**
- 고정환 외 (2010). *나로호(KSLV-I) 1차 비행시험 비행안전 운영*, 한국항공우주학회지 38(3)
- KARI 나로우주센터 소개: https://www.kari.re.kr/kor/contents/56
- KASA 민간기업 발사 가이드라인 (2026)

**임무보증**
- The Aerospace Corporation — Mission Assurance: https://aerospace.org/article/building-confidence-commercial-launch-national-security-space
- SimScale — Mars Climate Orbiter: https://www.simscale.com/blog/nasa-mars-climate-orbiter-metric/
